# opentelemetry_exporter_otlp_common

## Example 1: compression names

In [ ]:
# Example 1: Compression.from_str accepts the three OTLP names in any case and spacing, and refuses anything else
from opentelemetry.exporter.otlp.common.http import Compression

assert Compression.from_str(" GZIP ") is Compression.GZIP
assert Compression.from_str("deflate") is Compression.DEFLATE and Compression.from_str("None") is Compression.NONE
try:
    Compression.from_str("brotli")
except ValueError as error:
    print("Example 1: refused:", error)
else:
    raise AssertionError("an unknown compression was accepted")

## Example 2: the HTTP client compresses, sets Content-Encoding and reports success

In [ ]:
# Example 2: _OTLPHTTPClient with GZIP sends a gzip body + Content-Encoding through a recording transport
import gzip
from opentelemetry.exporter.http.transport._base import BaseHTTPResult, BaseHTTPTransport
from opentelemetry.exporter.otlp.common.http import Compression, _OTLPHTTPClient

class Result(BaseHTTPResult):
    def content(self): return b""
    def headers(self): return {}

class Recorder(BaseHTTPTransport):
    sent = []
    def request(self, method, url, *, headers=None, timeout=None, data=None):
        self.sent.append((method, url, dict(headers), data)); return Result(status_code=200, reason="OK")
    def is_connection_error(self, exception): return False
    def close(self): pass

client = _OTLPHTTPClient(Recorder(), "http://collector/v1/traces", "spans", compression=Compression.GZIP)
outcome = client.export(b"serialized spans")
method, url, headers, body = Recorder.sent[0]
assert outcome.success and outcome.status_code == 200 and method == "POST" and url.endswith("/v1/traces")
assert headers["Content-Encoding"] == "gzip" and gzip.decompress(body) == b"serialized spans"
print("Example 2:", outcome)

## Example 3: a 503 with Retry-After is retried; a 400 is not

In [ ]:
# Example 3: the client retries a retryable status (honouring Retry-After: 0) and gives up at once on a 400
from opentelemetry.exporter.http.transport._base import BaseHTTPResult, BaseHTTPTransport
from opentelemetry.exporter.otlp.common.http import _OTLPHTTPClient

class Result(BaseHTTPResult):
    def content(self): return b""
    def headers(self): return {"retry-after": "0"}

class Scripted(BaseHTTPTransport):
    def __init__(self, codes): self.codes, self.calls = list(codes), 0
    def request(self, method, url, **kwargs):
        self.calls += 1; return Result(status_code=self.codes.pop(0), reason="scripted")
    def is_connection_error(self, exception): return False
    def close(self): pass

retried, refused = Scripted([503, 503, 200]), Scripted([400])
assert _OTLPHTTPClient(retried, "http://c/v1/logs", "logs", timeout=5).export(b"x").success and retried.calls == 3
outcome = _OTLPHTTPClient(refused, "http://c/v1/logs", "logs", timeout=5).export(b"x")
assert not outcome.success and outcome.status_code == 400 and refused.calls == 1, outcome
print("Example 3: 503,503,200 ->", retried.calls, "calls, success | 400 ->", refused.calls, "call, failed")

## Example 4: metric temporality and histogram aggregation follow the OTLP environment variables

In [ ]:
# Example 4: the defaults are cumulative + explicit buckets; DELTA / exponential switch them; explicit choices win
import os
from opentelemetry.exporter.otlp.common._aggregation import _get_aggregation, _get_temporality
from opentelemetry.sdk.metrics import Counter, Histogram, UpDownCounter
from opentelemetry.sdk.metrics.export import AggregationTemporality as T
from opentelemetry.sdk.metrics.view import ExplicitBucketHistogramAggregation, ExponentialBucketHistogramAggregation

names = ("OTEL_EXPORTER_OTLP_METRICS_TEMPORALITY_PREFERENCE", "OTEL_EXPORTER_OTLP_METRICS_DEFAULT_HISTOGRAM_AGGREGATION")
saved = {name: os.environ.pop(name, None) for name in names}
try:
    assert _get_temporality(None)[Counter] is T.CUMULATIVE
    assert isinstance(_get_aggregation(None)[Histogram], ExplicitBucketHistogramAggregation)
    os.environ[names[0]], os.environ[names[1]] = "delta", "base2_exponential_bucket_histogram"
    delta = _get_temporality({Histogram: T.CUMULATIVE})
    assert delta[Counter] is T.DELTA and delta[UpDownCounter] is T.CUMULATIVE and delta[Histogram] is T.CUMULATIVE
    assert isinstance(_get_aggregation(None)[Histogram], ExponentialBucketHistogramAggregation)
finally:
    for name, value in saved.items():
        os.environ.pop(name, None) if value is None else os.environ.__setitem__(name, value)
print("Example 4: DELTA preference ->", {k.__name__: v.name for k, v in delta.items()})

## Example 5: end to end through the real urllib3 transport, with the exporter's self-metrics

In [ ]:
# Example 5: a deflate-compressed export reaches a local server; ExporterMetrics records 4 exported spans
import http.server, threading, zlib
from opentelemetry.exporter.http.transport._urllib3 import Urllib3HTTPTransport
from opentelemetry.exporter.otlp.common._exporter_metrics import create_exporter_metrics
from opentelemetry.exporter.otlp.common.http import Compression, _OTLPHTTPClient
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import InMemoryMetricReader

got = []
class Sink(http.server.BaseHTTPRequestHandler):
    def do_POST(self):
        got.append(zlib.decompress(self.rfile.read(int(self.headers["Content-Length"]))))
        self.send_response(200); self.send_header("Content-Length", "0"); self.end_headers()
    def log_message(self, *args): pass

server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), Sink)
threading.Thread(target=server.serve_forever, daemon=True).start()
endpoint = f"http://127.0.0.1:{server.server_address[1]}/v1/traces"
reader = InMemoryMetricReader()
metrics = create_exporter_metrics(None, "traces", endpoint, MeterProvider(metric_readers=[reader]), enabled=True)
with metrics.export_operation(4):
    outcome = _OTLPHTTPClient(Urllib3HTTPTransport(), endpoint, "spans", compression=Compression.DEFLATE).export(b"4 spans")
server.shutdown()
points = {m.name: m.data.data_points[0] for rm in reader.get_metrics_data().resource_metrics for sm in rm.scope_metrics for m in sm.metrics}
exported = points["otel.sdk.exporter.span.exported"]
assert outcome.success and got == [b"4 spans"] and exported.value == 4 and exported.attributes["server.address"] == "127.0.0.1"
print("Example 5:", outcome.status_code, got, "| exported spans:", exported.value, dict(exported.attributes))